# Ring-by-ring vs pooled ego-nets (EgoNEExT)

**Question.** EgoNEExT represents a node by the *bag* of nodes around it (its k-hop ego-net), summarised as one
distribution of structural features and embedded with optimal transport. A pooled bag mixes the centre's near and far
neighbours: a node one hop away counts the same as one three hops away. **Does keeping the hop rings separate,
[centre | ring 1 | ring 2 | ...], carry more label signal than the pooled bag at the same radius?**

This came out of the EgoNEExT team meeting (October 2026): split the ego-net into rings, represent each ring
separately, and concatenate, so that in the worst case the representation still contains the node's own features.

**What is compared**, on the same centres, splits and classifier (the paper's protocol):

| name | representation |
|---|---|
| `node` | the centre's own structural features (the paper's node-features baseline; ring 0 alone) |
| `pooled_k` | the paper's k-hop bag: all members within k hops, one distribution, approximate Wasserstein embedding |
| `ringmean_k` | centre features, then the mean feature vector of ring 1, ..., ring k, side by side, plus each ring's size |
| `ringblock_k` | NEExT `compute_ring_embeddings`: each ring embedded on its own (approximate Wasserstein), concatenated |
| `ringexact_k` | the same with exact Wasserstein (LOT) per ring, which keeps each ring's distribution shape, not only its mean |

All features use the **global scope**: each member contributes its full-graph feature values, so a ring answers
"who sits at distance r", and the pooled bag and the rings see exactly the same members.

**Why this is the right comparison.** Under the approximate embedding a pooled bag is (a linear map of) the mean of its
members, and that mean is a weighted average of the ring means. So `ringmean_k` contains everything `pooled_k` does and
more. `ringmean_k` is also what NEExT's `feature_vector_length = k + 1` computes (the hop-shell means), so this tests a
setting the paper never used (all its runs used length 1).

**Expectations, written before running** (state any change of plan in the README):
1. `ringmean_k` and `ringblock_k` are at least as good as `pooled_k` at the same k on every dataset, within noise.
2. The gain is largest where the pooled profile *falls* with k (dilution), because rings stop distant, uninformative
   members from drowning the near ones.
3. Exact per-ring Wasserstein adds little over ring means on these datasets (most of the paper's signal was in means).

**Caveat on what a gain means.** On homophilous graphs part of any neighbourhood score can be *position* rather than
structure (nearby train and test centres share members). The paper's region-holdout instrument separates the two; it is
not run here. Treat this notebook as the shared-split comparison, the first step.


## Setup

From the NEExT repository root on the `dev` branch:

```bash
uv sync --extra dev --extra experiments --extra advanced
uv run jupyter lab shared_experiments/egonext-ring-vs-pooled/ring_vs_pooled.ipynb
```

The datasets download on first use from the `anomalypoint/NEExT` collection on Hugging Face into `data/` next to this
notebook; full-graph features are cached in `outputs/`. Both folders are git-ignored.


In [ ]:
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

import NEExT as neext_pkg
from NEExT import NEExT
from NEExT.embeddings import member_rings
from NEExT.workbench.dataset_library import get_catalog_dataset

try:  # keep the BLAS / OpenMP pools at the configured size (call after all imports)
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

## Configuration

The two small datasets run in a few minutes. Roman Empire and Amazon Ratings need one full-graph feature pass each
(about 25 and 50 minutes on a laptop, then cached); uncomment them for the full study.


In [ ]:
DATASETS = {
    # catalog id: label column and number of sampled centres (None = every node), as in the paper
    "AIRPORTS_EUROPE": {"label": "activity_quartile", "n_centers": None},
    "WIKIPEDIA_ARTICLES_CHAMELEON": {"label": "traffic_quintile", "n_centers": 1000},
    # "ROMAN_EMPIRE": {"label": "syntactic_role", "n_centers": 3000},
    # "AMAZON_RATINGS": {"label": "rating_class", "n_centers": 3000},
}
MAX_K = 3                # radii k = 1..MAX_K
EXACT_MAX_K = 2          # ringexact_k for k <= this (exact LOT is O(n^3) per ring; 0 turns it off)
N_JOBS = 4               # CPU workers for features and XGBoost
FEATURES = ["all"]       # NEExT's 11 built-in structural features

CENTER_SEED, SPLIT_SEED, MODEL_SEED, EMBED_SEED, PERMUTATION_SEED = 13, 42, 42, 42, 7
N_SPLITS, TEST_SIZE, MIN_CLASS_COUNT = 10, 0.3, 10

HERE = Path.cwd()
DATA, OUT = HERE / "data", HERE / "outputs"
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
if threadpool_limits is not None:
    threadpool_limits(limits=N_JOBS)

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=HERE, text=True).strip()
    except Exception:
        return "unknown"

RUN_INFO = {"neext_version": neext_pkg.__version__, "neext_commit": git_commit(), "started": time.strftime("%Y-%m-%d %H:%M")}
RUN_INFO

## Data and centres

Same rules as the paper: structure and label only (native node attributes dropped), largest connected component,
classes with fewer than 10 sampled nodes removed, a stratified sample of centres with seed 13.


In [ ]:
nxt = NEExT(log_level="WARNING")

def load_dataset(catalog_id, label):
    catalog = get_catalog_dataset(catalog_id)
    frames = {}
    for key in ("nodes", "edges"):
        cache = DATA / catalog_id / f"{key}.csv"
        if not cache.exists():
            cache.parent.mkdir(parents=True, exist_ok=True)
            pd.read_csv(catalog.files[key]).to_csv(cache, index=False)
        frames[key] = pd.read_csv(cache)
    nodes = frames["nodes"][["node_id", label]]
    return nxt.load_single_graph_from_dfs(edges_df=frames["edges"], nodes_df=nodes, graph_type="igraph", filter_largest_component=True)

def node_table(collection, label, n_centers):
    graph = collection.graphs[0]
    table = pd.DataFrame([(n, graph.node_attributes[n][label]) for n in graph.nodes], columns=["node_id", "y_raw"])
    classes = sorted(table["y_raw"].unique(), key=str)
    table["y"] = table["y_raw"].map({c: i for i, c in enumerate(classes)})
    keep = table["y"].value_counts()
    table = table[table["y"].isin(keep[keep >= MIN_CLASS_COUNT].index)]
    if n_centers is not None and len(table) > n_centers:
        idx, _ = train_test_split(np.arange(len(table)), train_size=n_centers, random_state=CENTER_SEED, stratify=table["y"])
        table = table.iloc[idx]
        keep = table["y"].value_counts()
        table = table[table["y"].isin(keep[keep >= MIN_CLASS_COUNT].index)]
    return table.sort_values("node_id").reset_index(drop=True)

## Representations

Full-graph features are computed once per dataset (standardised over all nodes) and projected onto the members of
every bag. Ego-nets are built for the centres only.


In [ ]:
def source_features(catalog_id, collection):
    cache = OUT / f"{catalog_id}__features.parquet"
    if cache.exists():
        df = pd.read_parquet(cache)
        from NEExT.features import Features
        return Features(df, [c for c in df.columns if c not in ("node_id", "graph_id")])
    t0 = time.time()
    feats = nxt.compute_node_features(collection, feature_list=FEATURES, feature_vector_length=1, n_jobs=N_JOBS, show_progress=False)
    feats.features_df.to_parquet(cache)
    print(f"  full-graph features: {len(feats.feature_columns)} columns in {time.time() - t0:.0f}s")
    return feats

def by_centre(egonets, df):
    # graph_id (one ego-net per centre) -> node_id of the centre
    centre = {gid: node for gid, (_, node) in egonets.egonet_to_graph_node_mapping.items()}
    out = df.copy()
    out.insert(0, "node_id", out.pop("graph_id").map(centre))
    return out

def ring_means(egonets, projected, k):
    cols = projected.feature_columns
    table = projected.features_df.merge(member_rings(egonets), on=["node_id", "graph_id"])
    table = table[table["ring"] <= k]
    means = table.groupby(["graph_id", "ring"])[cols].mean().unstack("ring")
    means.columns = [f"{c}_ring{r}" for c, r in means.columns]
    sizes = table.groupby(["graph_id", "ring"]).size().unstack("ring")
    sizes.columns = [f"ring{r}_size" for r in sizes.columns]
    wide = pd.concat([means, sizes], axis=1)
    for r in range(k + 1):  # rings no member reaches: zeros, size 0
        for c in [f"{c}_ring{r}" for c in cols] + [f"ring{r}_size"]:
            if c not in wide:
                wide[c] = 0.0
    return wide.fillna(0.0).reset_index()

def build_representations(catalog_id, collection, centres):
    feats = source_features(catalog_id, collection)
    d = len(feats.feature_columns)
    gid = collection.graphs[0].graph_id
    reps = {"node": feats.features_df[feats.features_df["node_id"].isin(set(centres))].drop(columns=["graph_id"])}
    for k in range(1, MAX_K + 1):
        t0 = time.time()
        egonets = nxt.compute_k_hop_egonets(collection, k_hop=k, nodes_to_sample={gid: list(centres)}, sample_fraction=0.0)
        projected = egonets.project_source_features(feats)
        pooled = nxt.compute_graph_embeddings(egonets, projected, embedding_algorithm="approx_wasserstein", embedding_dimension=d, random_state=EMBED_SEED)
        reps[f"pooled_{k}"] = by_centre(egonets, pooled.embeddings_df)
        reps[f"ringmean_{k}"] = by_centre(egonets, ring_means(egonets, projected, k))
        blocks = nxt.compute_ring_embeddings(egonets, projected, max_ring=k, random_state=EMBED_SEED)
        reps[f"ringblock_{k}"] = by_centre(egonets, blocks.embeddings_df)
        if k <= EXACT_MAX_K:
            exact = nxt.compute_ring_embeddings(egonets, projected, max_ring=k, embedding_algorithm="wasserstein", random_state=EMBED_SEED)
            reps[f"ringexact_{k}"] = by_centre(egonets, exact.embeddings_df)
        sizes = pd.Series([len(g.nodes) for g in egonets.graphs])
        print(f"  k={k}: median bag {sizes.median():.0f} nodes (max {sizes.max()}), built in {time.time() - t0:.0f}s")
    return reps

## Evaluation

The paper's protocol: 10 stratified 70/30 splits (seeds 42 to 51) shared by every representation, so all comparisons
are paired; XGBoost (200 trees, depth 4); macro-F1 as the headline (accuracy kept). The permutation floor trains the
same pipeline on shuffled labels.


In [ ]:
def evaluate(name, rep, table, permute=False):
    merged = table[["node_id", "y"]].merge(rep, on="node_id", how="left", validate="one_to_one")
    X = merged.drop(columns=["node_id", "y"]).to_numpy(dtype=np.float64)
    if np.isnan(X).any():
        raise ValueError(f"{name}: missing rows for some centres")
    y = merged["y"].to_numpy()
    if permute:
        y = np.random.RandomState(PERMUTATION_SEED).permutation(y)
    classes = np.unique(y)
    y_enc = np.searchsorted(classes, y)
    rows = []
    for i in range(N_SPLITS):
        tr, te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=SPLIT_SEED + i, stratify=y)
        model = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="mlogloss", random_state=MODEL_SEED, n_jobs=N_JOBS)
        model.fit(X[tr], y_enc[tr])
        pred = model.predict(X[te])
        rows.append({"representation": name + ("_permuted" if permute else ""), "split": i, "n_columns": X.shape[1],
                     "macro_f1": f1_score(y_enc[te], pred, average="macro"), "accuracy": float(np.mean(pred == y_enc[te]))})
    return rows

## Run

In [ ]:
all_rows = []
for catalog_id, spec in DATASETS.items():
    t0 = time.time()
    print(catalog_id)
    collection = load_dataset(catalog_id, spec["label"])
    table = node_table(collection, spec["label"], spec["n_centers"])
    print(f"  {len(collection.graphs[0].nodes)} nodes, {len(table)} centres, {table['y'].nunique()} classes")
    reps = build_representations(catalog_id, collection, table["node_id"])
    rows = []
    for name, rep in reps.items():
        rows += evaluate(name, rep, table)
    rows += evaluate("node", reps["node"], table, permute=True)
    rows += evaluate(f"ringmean_{MAX_K}", reps[f"ringmean_{MAX_K}"], table, permute=True)
    for row in rows:
        row.update(dataset=catalog_id, **RUN_INFO)
    all_rows += rows
    print(f"  done in {time.time() - t0:.0f}s")

results = pd.DataFrame(all_rows)
results.to_csv(OUT / "results.csv", index=False)
print(f"{len(results)} rows -> {OUT / 'results.csv'}")

## Results

Macro-F1, mean ± standard deviation over the 10 shared splits.


In [ ]:
summary = (results.groupby(["dataset", "representation"])["macro_f1"].agg(["mean", "std"]).round(3)
           .assign(cell=lambda d: d["mean"].map("{:.3f}".format) + " ± " + d["std"].map("{:.3f}".format))["cell"]
           .unstack("dataset"))
order = ["node", "node_permuted"] + [f"{m}_{k}" for k in range(1, MAX_K + 1) for m in ("pooled", "ringmean", "ringblock", "ringexact")] + [f"ringmean_{MAX_K}_permuted"]
summary.reindex([r for r in order if r in summary.index])

### Paired tests against the pooled bag at the same radius

Difference in macro-F1 (ring minus pooled), averaged over splits, with a two-sided paired t-test over the 10 splits.
With this many comparisons, read p-values near 0.05 with caution.


In [ ]:
def paired(dataset, a, b):
    sa = results[(results.dataset == dataset) & (results.representation == a)].sort_values("split")["macro_f1"].to_numpy()
    sb = results[(results.dataset == dataset) & (results.representation == b)].sort_values("split")["macro_f1"].to_numpy()
    return {"dataset": dataset, "comparison": f"{a} - {b}", "diff": round(float((sa - sb).mean()), 4), "p": round(float(ttest_rel(sa, sb).pvalue), 4)}

tests = []
for dataset in DATASETS:
    for k in range(1, MAX_K + 1):
        for ring in ("ringmean", "ringblock", "ringexact"):
            if f"{ring}_{k}" in set(results.representation):
                tests.append(paired(dataset, f"{ring}_{k}", f"pooled_{k}"))
        tests.append(paired(dataset, f"ringmean_{k}", "node"))
pd.DataFrame(tests)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5 * len(DATASETS), 4), squeeze=False)
ks = list(range(1, MAX_K + 1))
for ax, dataset in zip(axes[0], DATASETS):
    sub = results[results.dataset == dataset].groupby("representation")["macro_f1"].mean()
    for method, style in [("pooled", "o-"), ("ringmean", "s-"), ("ringblock", "^-"), ("ringexact", "d-")]:
        present = [k for k in ks if f"{method}_{k}" in sub]
        if present:
            ax.plot(present, [sub[f"{method}_{k}"] for k in present], style, label=method)
    ax.axhline(sub["node"], color="green", ls=":", label="node")
    ax.axhline(sub["node_permuted"], color="grey", ls="--", label="permutation floor")
    ax.set_title(dataset)
    ax.set_xlabel("radius k")
    ax.set_xticks(ks)
    ax.set_ylabel("macro-F1")
axes[0][0].legend()
fig.tight_layout()
fig.savefig(OUT / "ring_vs_pooled.pdf")

## Reading the results

- **Expectation 1** holds if every `ring*_k` is at or above `pooled_k` (difference ≥ 0 or not significant).
- **Expectation 2**: compare the slope of `pooled` with k against the ring curves; rings should not fall where pooled
  falls.
- **Expectation 3**: `ringexact_k` vs `ringmean_k`.
- `ringmean_k` against `node` says whether the neighbourhood adds anything at all once kept apart from the centre.
- Both permuted rows must sit at chance; if they do not, look for leakage before reading anything else.

Write the numbers and the reading into the README of this folder (section "Findings"), with the commit hash printed
in the configuration cell.
